In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import glob
import os

In [ ]:
# ==========================================
# 1. 定義模型架構 (Encoder + CTM)
# ==========================================

class IMUEncoder(nn.Module):
    """將 6軸 x 200點 的原始數據，壓縮成特徵向量"""
    def __init__(self, in_channels=6, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # 強制壓縮成長度 1
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

class NeuronLevelModel(nn.Module):
    def __init__(self, channels, history_len):
        super().__init__()
        self.history_len = history_len
        self.temporal = nn.Conv1d(channels, channels, kernel_size=history_len, groups=channels)
        self.act = nn.SiLU()
    
    def forward(self, history):
        # history: [B, C, T]
        if history.shape[2] < self.history_len:
            pad = self.history_len - history.shape[2]
            history = torch.nn.functional.pad(history, (pad, 0))
        recent = history[:, :, -self.history_len:]
        return self.act(self.temporal(recent).squeeze(-1))

class ContinuousThoughtMachine(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64, max_ticks=5):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.max_ticks = max_ticks
        
        self.encoder = IMUEncoder(input_dim, hidden_dim)
        self.nlm = NeuronLevelModel(hidden_dim, history_len=3)
        
        # 簡化版的 Sync Head (直接用 Linear 模擬同步注意力)
        self.sync_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, batch_first=True)
        
        self.gate = nn.GRUCell(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2) # 2類: 累 vs 不累

    def forward(self, x):
        # x: [Batch, 6, Seq_Len]
        features = self.encoder(x) # [B, H]
        
        state = features
        history = features.unsqueeze(2) # [B, H, 1]
        outputs = []
        
        for t in range(self.max_ticks):
            # 1. NLM 思考
            nlm_out = self.nlm(history)
            
            # 2. Sync (Attention between NLM thought and Original Input)
            # Query=NLM, Key=Input, Value=Input
            attn_out, _ = self.sync_attention(
                nlm_out.unsqueeze(1), 
                features.unsqueeze(1), 
                features.unsqueeze(1)
            )
            attn_out = attn_out.squeeze(1)
            
            # 3. Update State
            state = self.gate(attn_out, state)
            
            # 4. Record Output
            logits = self.classifier(state)
            outputs.append(logits)
            
            # Update History
            history = torch.cat([history, state.unsqueeze(2)], dim=2)
            if history.shape[2] > 10: history = history[:, :, -10:]
            
        return torch.stack(outputs, dim=1) # [Batch, Ticks, 2]

In [ ]:
# ==========================================
# 2. 資料處理 (Dataset & Sliding Window)
# ==========================================

class FatigueDataset(Dataset):
    def __init__(self, root_dir, window_size=200, step=50):
        self.samples = []
        self.labels = []
        
        # 讀取 "不累" (Label 0)
        self._load_csvs(os.path.join(root_dir, 'not_tired/*.csv'), 0, window_size, step)
        # 讀取 "累" (Label 1)
        self._load_csvs(os.path.join(root_dir, 'tired/*.csv'), 1, window_size, step)
        
        print(f"Dataset Loaded: {len(self.samples)} samples.")
        if len(self.samples) > 0:
            print(f"Sample shape: {self.samples[0].shape}")

    def _load_csvs(self, path_pattern, label, w_size, step):
        files = glob.glob(path_pattern)
        for f in files:
            try:
                # 假設 CSV 沒有 header，若有請改 header=0
                df = pd.read_csv(f, header=None) 
                data = df.values # numpy array
                
                # 簡單的標準化 (Z-Score Normalization) 
                # 實際專案建議算出全域 Mean/Std，這裡先做單檔標準化
                data = (data - data.mean(axis=0)) / (data.std(axis=0) + 1e-6)
                
                # 滑動視窗切片 (Sliding Window)
                # 數據形狀: [Time, 6] -> 轉置為 [6, Time]
                data = data.T 
                
                num_points = data.shape[1]
                for i in range(0, num_points - w_size, step):
                    window = data[:, i : i + w_size]
                    self.samples.append(torch.FloatTensor(window))
                    self.labels.append(label)
            except Exception as e:
                print(f"Error reading {f}: {e}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], self.labels[idx]

In [ ]:
# ==========================================
# 3. 訓練流程 (Main Loop)
# ==========================================

def train():
    # 設定參數
    WINDOW_SIZE = 300  # 假設 100Hz 取 2秒
    BATCH_SIZE = 100
    LR = 0.001
    EPOCHS = 20
    
    # 準備資料
    dataset = FatigueDataset('./data', window_size=WINDOW_SIZE)
    if len(dataset) == 0:
        print("錯誤：找不到資料，請檢查 data 資料夾")
        return

    # 切分訓練/驗證集 (80% / 20%)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_data, val_data = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
    
    # 初始化模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = ContinuousThoughtMachine(max_ticks=5).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    
    # 開始訓練
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        correct = 0
        total_samples = 0
        
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            
            optimizer.zero_grad()
            
            # Forward: 取得所有 Ticks 的輸出 [Batch, Ticks, Classes]
            outputs = model(X) 
            
            # 計算累積 Loss (Accumulated Loss)
            loss = 0
            for t in range(model.max_ticks):
                # 每個時刻的輸出都要算 loss
                loss += criterion(outputs[:, t, :], y)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # 計算最後一個 Tick 的準確率
            final_pred = outputs[:, -1, :].argmax(dim=1)
            correct += (final_pred == y).sum().item()
            total_samples += y.size(0)
            
        avg_loss = total_loss / len(train_loader)
        acc = correct / total_samples * 100
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {acc:.2f}%")
        
    # 儲存模型
    torch.save(model.state_dict(), "ctm_fatigue_model.pth")
    print("模型已儲存為 ctm_fatigue_model.pth")

if __name__ == "__main__":
    train()